# Tutorial 53: Creation of Longitudinal Section - and Edge Dataframe

This Example demonstrates the capabilities of the class Dataframes_SIR3S_Model that extends SIR3S_Model be abilities to work directley with pandas dataframes. It is shown how to create dataframes containing information that does not concern individual elements types such as Nodes, Pipes, etc. but instead concerning more abstract SIR 3S data such as longitudinal sections or concatenations of multiple element types like hydraulic edges.

# Toolkit Release

In [27]:
#pip install 

# Imports

## SIR 3S Toolkit

### Regular Import/Init

In [28]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [29]:
from sir3stoolkit.core import wrapper

In [30]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [31]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

[2026-05-25 14:18:21,782] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-05-25 14:18:21,785] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


### Additional Import/Init for Dataframes class

In [32]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [33]:
s3s = SIR3S_Model_Dataframes()

[2026-05-25 14:18:22,052] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


## Additional

In [34]:
import pandas as pd
from shapely.geometry import Point
import re
import folium
from folium.plugins import HeatMap
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import contextily as cx

# Open Model

In [35]:
s3s.OpenModel(dbName=r"Toolkit_Tutorial53_Model.db3",
              providerType=s3s.ProviderTypes.SQLite,
              Mid="M-1-0-1",
              saveCurrentlyOpenModel=False,
              namedInstance="",
              userID="",
              password="")

[2026-05-25 14:18:48,025] INFO in sir3stoolkit.core.wrapper: Model is open for further operation


# Generate Non-Element Dataframes

## Edge dataframe

We can use the function [generate_edge_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_edge_dataframe) to obtain a dataframe containing all edge objects in the model. Below is a list of all element types, whose representatives that will be included in the dataframe, if present in the model.

In [36]:
edge_types = [
    'Pipe', 'Valve', 'SafetyValve', 'PressureRegulator', 'DifferentialRegulator',
    'FlapValve', 'PhaseSeparation', 'FlowControlUnit', 'ControlValve', 'Pump',
    'DistrictHeatingConsumer', 'DistrictHeatingFeeder', 'Compressor', 'HeaterCooler', 'HeatFeederConsumerStation'
]

HeatExchanger is not supported.

### Parametrization 1: Default

If we don't request any properties, we will only get [tk, Fkcont, geometry, fkKI, fkKK, element_type, L] as df columns.

In [37]:
(s3s.generate_edge_dataframe()).head(5)

[2026-05-25 14:18:48,582] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] Generating dataframe for ObjectTypes.Pipe ...
[2026-05-25 14:18:48,608] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe
[2026-05-25 14:18:48,615] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-05-25 14:18:48,623] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 1 model_data properties.
[2026-05-25 14:18:48,628] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Fkcont'], geometry, end nodes...
[2026-05-25 14:18:49,226] INFO in sir3stoolkit.mantle.dataframes: [model_data] 2 non-empty end node columns were created.
[2026-05-25 14:18:49,340] INFO in sir3stoolkit.mantle.dataframes: [model_data] Transforming DataFrame to GeoDataFrame successful with EPSG: 25832
[2026-05-25 14:18:49,340] INFO in sir3sto

,Fkcont,L,element type,fkKI,fkKK,geometry,tk
0,5029128874972463118,0,Pipe,4730066059089961857,4917189080965035120,"LINESTRING (714262.483 5578857.42, 714269.543 ...",4614463970292122863
1,5029128874972463118,0,Pipe,5129584372458662150,5332825919690090061,"LINESTRING (713738.297 5579219.902, 713793.23 ...",4615723899944629797
2,5029128874972463118,0,Pipe,5070795580168283912,5725848577942138606,"LINESTRING (713650.613 5578990.488, 713649.498...",4621030304810285220
3,5029128874972463118,0,Pipe,5106194195554624313,5416743601805578486,"LINESTRING (713283.932 5578718.123, 713299.538...",4621904482639719098
4,5029128874972463118,0,Pipe,5395214589547810761,5186627743244018598,"LINESTRING (713231.461 5579074.621, 713266.467...",4623301797443553869


### Parametrization 2: Properties

We can request additional properties. Here we can mix model data and result properties in the same param. 

If a property is not defined for all edge element types, a column for that property will be created, but only populated, where values exists.

In [38]:
(s3s.generate_edge_dataframe(properties=["Hal", "Kvr", "JV", "PVEC", "MVEC"])).head(5) # use last simulation timestamp

[2026-05-25 14:18:52,611] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] Generating dataframe for ObjectTypes.Pipe ...
[2026-05-25 14:18:52,616] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe
[2026-05-25 14:18:52,622] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-05-25 14:18:52,627] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 3 model_data properties.
[2026-05-25 14:18:52,630] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Hal', 'Kvr', 'Fkcont'], geometry, end nodes...
[2026-05-25 14:18:53,318] INFO in sir3stoolkit.mantle.dataframes: [model_data] 2 non-empty end node columns were created.
[2026-05-25 14:18:53,419] INFO in sir3stoolkit.mantle.dataframes: [model_data] Transforming DataFrame to GeoDataFrame successful with EPSG: 25832
[2026-05-25 14:18:53,422] I

,Fkcont,Hal,JV,Kvr,L,MVEC_end,MVEC_sequence,MVEC_start,PVEC_end,PVEC_sequence,PVEC_start,element type,fkKI,fkKK,geometry,tk
0,5029128874972463118,0.0,0.0,2.0,0,2.837623e-10,"(2.837623e-10, 2.837623e-10)",2.837623e-10,1.688630,"(1.615055, 1.68863)",1.615055,Pipe,4730066059089961857,4917189080965035120,"LINESTRING (714262.483 5578857.42, 714269.543 ...",4614463970292122863
1,5029128874972463118,0.0,0.0,1.0,0,-2.983143e-10,"(-2.983143e-10, -2.983143e-10, -2.983143e-10, ...",-2.983143e-10,3.590000,"(3.304535, 3.345315, 3.386095, 3.426875, 3.467...",3.304535,Pipe,5129584372458662150,5332825919690090061,"LINESTRING (713738.297 5579219.902, 713793.23 ...",4615723899944629797
2,5029128874972463118,0.0,0.211137,2.0,0,-4.251046e+00,"(-4.251046, -4.251046)",-4.251046e+00,2.198520,"(2.210225, 2.19852)",2.210225,Pipe,5070795580168283912,5725848577942138606,"LINESTRING (713650.613 5578990.488, 713649.498...",4621030304810285220
3,5029128874972463118,0.0,0.184983,1.0,0,-2.413726e+01,"(-24.13726, -24.13726, -24.13726, -24.13726)",-2.413726e+01,4.214520,"(4.1376, 4.16324, 4.18888, 4.21452)",4.137600,Pipe,5106194195554624313,5416743601805578486,"LINESTRING (713283.932 5578718.123, 713299.538...",4621904482639719098
4,5029128874972463118,0.0,0.0,2.0,0,1.737135e-10,"(1.737135e-10, 1.737135e-10, 1.737135e-10, 1.7...",1.737135e-10,2.328365,"(2.304045, 2.30891, 2.313775, 2.318635, 2.3235...",2.304045,Pipe,5395214589547810761,5186627743244018598,"LINESTRING (713231.461 5579074.621, 713266.467...",4623301797443553869


The resulting df now includes model data properties such as "Hal" or "Kvr", but also result properties such as "JV" or result vector properties along pipe interior points like "PVEC" or "MVEC", which are split into the columns PVEC_start, PVEC_end, and PVEC_sequence = (PVEC_start, PVEC_2, PVEC_3, ..., PVEC_end).

As we didn't specify a timestamp, the result values are obtained for the stationary timestamp (s3s.GetTimeStamps()[1]).

Note that typical int cols like "Kvr" are float here. This happens due to the merge of many dataframes, some with NaN values. 

### Parametrization 3: Properties + Timestamps

In [39]:
simulation_timestamps = s3s.GetTimeStamps()[0]

In [40]:
simulation_timestamps

['2023-02-13 00:00:00.000 +01:00',
 '2023-02-13 01:00:00.000 +01:00',
 '2023-02-13 02:00:00.000 +01:00',
 '2023-02-13 03:00:00.000 +01:00',
 '2023-02-13 04:00:00.000 +01:00',
 '2023-02-13 05:00:00.000 +01:00',
 '2023-02-13 06:00:00.000 +01:00',
 '2023-02-13 07:00:00.000 +01:00',
 '2023-02-13 08:00:00.000 +01:00',
 '2023-02-13 09:00:00.000 +01:00',
 '2023-02-13 10:00:00.000 +01:00',
 '2023-02-13 11:00:00.000 +01:00',
 '2023-02-13 12:00:00.000 +01:00',
 '2023-02-13 13:00:00.000 +01:00',
 '2023-02-13 14:00:00.000 +01:00',
 '2023-02-13 15:00:00.000 +01:00',
 '2023-02-13 16:00:00.000 +01:00',
 '2023-02-13 17:00:00.000 +01:00',
 '2023-02-13 18:00:00.000 +01:00',
 '2023-02-13 19:00:00.000 +01:00',
 '2023-02-13 20:00:00.000 +01:00',
 '2023-02-13 21:00:00.000 +01:00',
 '2023-02-13 22:00:00.000 +01:00',
 '2023-02-13 23:00:00.000 +01:00',
 '2023-02-14 00:00:00.000 +01:00']

In [41]:
timestamp_idx = 8

In [42]:
timestamp_str = simulation_timestamps[timestamp_idx]

In [43]:
timestamp_str

'2023-02-13 08:00:00.000 +01:00'

In [44]:
(s3s.generate_edge_dataframe(properties=["Hal", "Kvr", "JV", "PVEC", "MVEC"]
                             ,timestamp=timestamp_idx)).head(5)

[2026-05-25 14:18:58,162] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] Generating dataframe for ObjectTypes.Pipe ...
[2026-05-25 14:18:58,168] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe
[2026-05-25 14:18:58,175] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-05-25 14:18:58,180] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 3 model_data properties.
[2026-05-25 14:18:58,182] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Hal', 'Kvr', 'Fkcont'], geometry, end nodes...
[2026-05-25 14:18:59,502] INFO in sir3stoolkit.mantle.dataframes: [model_data] 2 non-empty end node columns were created.
[2026-05-25 14:18:59,896] INFO in sir3stoolkit.mantle.dataframes: [model_data] Transforming DataFrame to GeoDataFrame successful with EPSG: 25832
[2026-05-25 14:18:59,896] I

,Fkcont,Hal,JV,Kvr,L,MVEC_end,MVEC_sequence,MVEC_start,PVEC_end,PVEC_sequence,PVEC_start,element type,fkKI,fkKK,geometry,tk
0,5029128874972463118,0.0,0.0,2.0,0,1.382432e-10,"(1.382432e-10, 1.382432e-10)",1.382432e-10,1.688630,"(1.615055, 1.68863)",1.615055,Pipe,4730066059089961857,4917189080965035120,"LINESTRING (714262.483 5578857.42, 714269.543 ...",4614463970292122863
1,5029128874972463118,0.0,0.0,1.0,0,-2.983143e-10,"(-2.983143e-10, -2.983143e-10, -2.983143e-10, ...",-2.983143e-10,3.847980,"(3.562515, 3.603295, 3.644075, 3.684855, 3.725...",3.562515,Pipe,5129584372458662150,5332825919690090061,"LINESTRING (713738.297 5579219.902, 713793.23 ...",4615723899944629797
2,5029128874972463118,0.0,0.667982,2.0,0,-7.826131e+00,"(-7.826131, -7.826131)",-7.826131e+00,2.805950,"(2.81585, 2.80595)",2.815850,Pipe,5070795580168283912,5725848577942138606,"LINESTRING (713650.613 5578990.488, 713649.498...",4621030304810285220
3,5029128874972463118,0.0,0.609716,1.0,0,-4.478037e+01,"(-44.78037, -44.78037, -44.78037, -44.78037)",-4.478037e+01,4.874130,"(4.788045, 4.81674, 4.845435, 4.87413)",4.788045,Pipe,5106194195554624313,5416743601805578486,"LINESTRING (713283.932 5578718.123, 713299.538...",4621904482639719098
4,5029128874972463118,0.0,0.0,2.0,0,-1.737135e-10,"(-1.737135e-10, -1.737135e-10, -1.737135e-10, ...",-1.737135e-10,2.712575,"(2.68805, 2.692955, 2.69786, 2.702765, 2.70767...",2.688050,Pipe,5395214589547810761,5186627743244018598,"LINESTRING (713231.461 5579074.621, 713266.467...",4623301797443553869


In [45]:
(s3s.generate_edge_dataframe(properties=["Hal", "Kvr", "JV", "PVEC", "MVEC"]
                             ,timestamp=timestamp_str)).head(5)

[2026-05-25 14:19:06,357] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] Generating dataframe for ObjectTypes.Pipe ...
[2026-05-25 14:19:06,365] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe
[2026-05-25 14:19:06,371] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-05-25 14:19:06,376] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 3 model_data properties.
[2026-05-25 14:19:06,378] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Hal', 'Kvr', 'Fkcont'], geometry, end nodes...
[2026-05-25 14:19:07,435] INFO in sir3stoolkit.mantle.dataframes: [model_data] 2 non-empty end node columns were created.
[2026-05-25 14:19:07,607] INFO in sir3stoolkit.mantle.dataframes: [model_data] Transforming DataFrame to GeoDataFrame successful with EPSG: 25832
[2026-05-25 14:19:07,618] I

,Fkcont,Hal,JV,Kvr,L,MVEC_end,MVEC_sequence,MVEC_start,PVEC_end,PVEC_sequence,PVEC_start,element type,fkKI,fkKK,geometry,tk
0,5029128874972463118,0.0,0.0,2.0,0,1.382432e-10,"(1.382432e-10, 1.382432e-10)",1.382432e-10,1.688630,"(1.615055, 1.68863)",1.615055,Pipe,4730066059089961857,4917189080965035120,"LINESTRING (714262.483 5578857.42, 714269.543 ...",4614463970292122863
1,5029128874972463118,0.0,0.0,1.0,0,-2.983143e-10,"(-2.983143e-10, -2.983143e-10, -2.983143e-10, ...",-2.983143e-10,3.847980,"(3.562515, 3.603295, 3.644075, 3.684855, 3.725...",3.562515,Pipe,5129584372458662150,5332825919690090061,"LINESTRING (713738.297 5579219.902, 713793.23 ...",4615723899944629797
2,5029128874972463118,0.0,0.667982,2.0,0,-7.826131e+00,"(-7.826131, -7.826131)",-7.826131e+00,2.805950,"(2.81585, 2.80595)",2.815850,Pipe,5070795580168283912,5725848577942138606,"LINESTRING (713650.613 5578990.488, 713649.498...",4621030304810285220
3,5029128874972463118,0.0,0.609716,1.0,0,-4.478037e+01,"(-44.78037, -44.78037, -44.78037, -44.78037)",-4.478037e+01,4.874130,"(4.788045, 4.81674, 4.845435, 4.87413)",4.788045,Pipe,5106194195554624313,5416743601805578486,"LINESTRING (713283.932 5578718.123, 713299.538...",4621904482639719098
4,5029128874972463118,0.0,0.0,2.0,0,-1.737135e-10,"(-1.737135e-10, -1.737135e-10, -1.737135e-10, ...",-1.737135e-10,2.712575,"(2.68805, 2.692955, 2.69786, 2.702765, 2.70767...",2.688050,Pipe,5395214589547810761,5186627743244018598,"LINESTRING (713231.461 5579074.621, 713266.467...",4623301797443553869


The above two dataframes were requested for the same timestamp, therefore they are identical

In [46]:
df = s3s.generate_element_model_data_dataframe(s3s.ObjectTypes.Pipe)

[2026-05-25 14:19:14,887] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe
[2026-05-25 14:19:14,899] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-05-25 14:19:14,908] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.Pipe.
[2026-05-25 14:19:14,911] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 46 model_data properties.
[2026-05-25 14:19:14,919] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Name', 'FkdtroRowd', 'Fkltgr', 'Fkstrasse', 'L', 'Lzu', 'Rau', 'Jlambs', 'Lambda0', 'Zein', 'Zaus', 'Zuml', 'Asoll', 'Indschall', 'Baujahr', 'Hal', 'Fkcont', 'Fk2lrohr', 'Beschreibung', 'Idreferenz', 'Iplanung', 'Kvr', 'LineWidthMM', 'DottedLine', 'DN', 'Di', 'KvrKlartext', 'HasClosedNSCHs', '

## AGSN (Longitudinal Sections)

We can use the method [generate_longitudinal_section_dataframes()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_longitudinal_section_dataframes) to obtain a list indexed by (lfdnr-1) filled with size-2-tuples with 0 - Supply and 1 - Return..

In [47]:
dfs = s3s.generate_longitudinal_section_dataframes()

[2026-05-25 14:19:24,942] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.AGSN_HydraulicProfile
[2026-05-25 14:19:24,946] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 3 element(s) of element type ObjectTypes.AGSN_HydraulicProfile.
[2026-05-25 14:19:24,950] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.AGSN_HydraulicProfile.
[2026-05-25 14:19:24,953] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 9 model_data properties.
[2026-05-25 14:19:24,955] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Name', 'Lfdnr', 'Aktiv', 'AllNodesAndLinks', 'ObjsString', 'MainWay', 'Tk', 'Pk', 'InVariant']...
[2026-05-25 14:19:25,004] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (3, 10)
[2026-05-25 14:19:25,010] INFO in sir3stoolkit.m

In [48]:
dfs[0][0].head() # Lfdnr 1, Vl

,tk,Name,FkdtroRowd,Fkltgr,Fkstrasse,L,Lzu,Rau,Jlambs,Lambda0,Zein,Zaus,Zuml,Asoll,Indschall,Baujahr,Hal,Fkcont,Fk2lrohr,Beschreibung,Idreferenz,Iplanung,Kvr,LineWidthMM,DottedLine,DN,Di,KvrKlartext,HasClosedNSCHs,Tk,Pk,InVariant,Xkor,Ykor,GeometriesDiffer,bz.Fk,bz.Qsvb,bz.Irtrenn,bz.Leckstatus,bz.Leckstart,bz.Leckend,bz.Leckort,bz.Leckmenge,bz.Imptnz,bz.Zvlimptnz,bz.Kantenzv,bz.ITrennWithNSCH,geometry,fkKI,fkKK,PipeTable: Table Name,PipeTable: Name,PipeTable: Fk,PipeTable: Dn,PipeTable: Di,PipeTable: Da,PipeTable: S,PipeTable: Wsteig,PipeTable: Wtiefe,PipeTable: Kt,PipeTable: Pn,PipeTable: Ausfallzeit,PipeTable: Reparatur,PipeTable: Rehabilitation,PipeTable: Tk,PipeTable: Pk,PipeTable: InVariant,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_start,MVEC_end,MVEC_sequence,PDAMPF,PHR,PMIN,PVEC_start,PVEC_end,PVEC_sequence,PVECMAX_INST_start,PVECMAX_INST_end,PVECMAX_INST_sequence,PVECMIN_INST_start,PVECMIN_INST_end,PVECMIN_INST_sequence,QMAV,QMI,QMK,RHOI,RHOK,RHOVEC_start,RHOVEC_end,RHOVEC_sequence,SVEC_start,SVEC_end,SVEC_sequence,TI,TK,TTRVEC_start,TTRVEC_end,TTRVEC_sequence,TVEC_start,TVEC_end,TVEC_sequence,VAV,VI,VK,VOLDA,WVL,ZVEC_start,ZVEC_end,ZVEC_sequence,l_sum,AGSN_Lfdnr,AGSN_Name
0,5691533564979419761,Rohr V-E0 V-K1683S,4816511167455310973,4779752876656844188,5204829332281547346,15.351700,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5025945677694931826,OSM: Knoten 476971238 -> Knoten 299394923; Län...,39785520,0,1,0.005,0,350,345.6,Vorlauf,,5691533564979419761,5691533564979419761,False,713619.921383,5.578219e+06,False,5691533564979419761,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713619.921 5578218.954, 713614.649...",5398100694284104779,4825391580467484032,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,0.0,0.004844,43.12018,0.661968,0.0,0.0,0.152727,79.74764,79.74764,"(79.74764, 79.74764, 79.74764)",0.701074,0.002345,5.800529,5.878635,5.800530,"(5.878635, 5.83958, 5.80053)",5.878635,5.800530,"(5.878635, 5.83958, 5.80053)",5.878635,5.800530,"(5.878635, 5.83958, 5.80053)",287.0915,287.0915,287.0915,965.7,965.7012,965.7000,965.7012,"(965.7, 965.7006, 965.7012)",0.0,15.351700,"(0.0, 7.67585, 15.3517)",89.99999,89.99802,0.000000,0.004844,"(0.0, 0.002422066, 0.004844132)",90.00000,89.99802,"(90.0, 89.99902, 89.99802)",0.880315,0.880315,0.880314,0.0,10082.88,541.49,542.29,"(541.49, 541.89, 542.29)",15.351701,1,Längsschnitt
1,5048873293262650113,Rohr V-K1683S V-K1693S,4816511167455310973,4779752876656844188,5204829332281547346,12.508950,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5216742060270992761,OSM: Knoten 299394923 -> Knoten 4105649557; Lä...,39785520,0,1,0.005,0,350,345.6,Vorlauf,,5048873293262650113,5048873293262650113,False,713614.648712,5.578233e+06,False,5048873293262650113,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713614.649 5578233.372, 713614.465...",4825391580467484032,5180617780362861593,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,0.0,0.003947,43.58081,0.54515,0.0,0.0,0.152727,79.74764,79.74764,"(79.74764, 79.74764, 79.74764)",0.701027,0.00191,5.732326,5.800530,5.732325,"(5.80053, 5.76643, 5.732325)",5.800530,5.732325,"(5.80053, 5.76643, 5.732325)",5.800530,5.732325,"(5.80053, 5.76643, 5.732325)",287.0915,287.0915,287.0915,965.7012,965.7021,965.7012,965.7021,"(965.7012, 965.7017, 965.7021)",0.0,12.508950,"(0.0, 6.254475, 12.50895)",89.99802,89.9964,0.004844,0.008791,"(0.004844132, 0.006817694, 0.008791257)",89.99802,89.99640,"(89.99802, 89.99722, 89.9964)",0.880314,0.880314,0.880313,0.0,10081.81,542.29,542.99,"(542.29, 542.64, 542.99)",27.860648,1,Längsschnitt
2,5715081934973525403,Rohr V-K2163S V-K1693S,4816511167455310973,4779752876656844188,5204829332281547346,6.265505,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,4919359344392474301,OSM: Knoten 299394922 -> Knoten 4105649557; Lä...,595926249,0,1,0.005,0,350,345.6,Vorlauf,,5715081934973525403,5715081934973525403,False,713614.3

In [49]:
dfs[0][1].head() # Lfdnr 1 RL

,tk,Name,FkdtroRowd,Fkltgr,Fkstrasse,L,Lzu,Rau,Jlambs,Lambda0,Zein,Zaus,Zuml,Asoll,Indschall,Baujahr,Hal,Fkcont,Fk2lrohr,Beschreibung,Idreferenz,Iplanung,Kvr,LineWidthMM,DottedLine,DN,Di,KvrKlartext,HasClosedNSCHs,Tk,Pk,InVariant,Xkor,Ykor,GeometriesDiffer,bz.Fk,bz.Qsvb,bz.Irtrenn,bz.Leckstatus,bz.Leckstart,bz.Leckend,bz.Leckort,bz.Leckmenge,bz.Imptnz,bz.Zvlimptnz,bz.Kantenzv,bz.ITrennWithNSCH,geometry,fkKI,fkKK,PipeTable: Table Name,PipeTable: Name,PipeTable: Fk,PipeTable: Dn,PipeTable: Di,PipeTable: Da,PipeTable: S,PipeTable: Wsteig,PipeTable: Wtiefe,PipeTable: Kt,PipeTable: Pn,PipeTable: Ausfallzeit,PipeTable: Reparatur,PipeTable: Rehabilitation,PipeTable: Tk,PipeTable: Pk,PipeTable: InVariant,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_start,MVEC_end,MVEC_sequence,PDAMPF,PHR,PMIN,PVEC_start,PVEC_end,PVEC_sequence,PVECMAX_INST_start,PVECMAX_INST_end,PVECMAX_INST_sequence,PVECMIN_INST_start,PVECMIN_INST_end,PVECMIN_INST_sequence,QMAV,QMI,QMK,RHOI,RHOK,RHOVEC_start,RHOVEC_end,RHOVEC_sequence,SVEC_start,SVEC_end,SVEC_sequence,TI,TK,TTRVEC_start,TTRVEC_end,TTRVEC_sequence,TVEC_start,TVEC_end,TVEC_sequence,VAV,VI,VK,VOLDA,WVL,ZVEC_start,ZVEC_end,ZVEC_sequence,l_sum,AGSN_Lfdnr,AGSN_Name
0,5025945677694931826,Rohr R-E0 R-K4163S,4816511167455310973,4779752876656844188,5204829332281547346,15.351700,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5691533564979419761,OSM: Knoten 476971238 -> Knoten 299394923; Län...,39785520,0,2,0.005,0,350,345.6,Rücklauf,,5025945677694931826,5025945677694931826,False,713621.921383,5.578219e+06,False,5025945677694931826,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713621.921 5578218.954, 713616.649...",5160850648779674898,4851678476955084637,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,0.0,0.004935,26.63305,0.408863,0.0,0.0,0.154884,-79.74764,-79.74764,"(-79.74764, -79.74764, -79.74764)",0.197685,0.002378,3.953543,4.025455,3.953540,"(4.025455, 3.9895, 3.95354)",4.025455,3.953540,"(4.025455, 3.9895, 3.95354)",4.025455,3.953540,"(4.025455, 3.9895, 3.95354)",-287.0915,-287.0915,-287.0915,983.7845,983.7839,983.7845,983.7839,"(983.7845, 983.7842, 983.7839)",0.0,15.351700,"(0.0, 7.67585, 15.3517)",59.83105,59.83229,0.316736,0.311801,"(0.3167359, 0.3142685, 0.311801)",59.83105,59.83231,"(59.83105, 59.83167, 59.83231)",-0.864133,-0.864133,-0.864133,0.0,0.0,541.43,542.20,"(541.43, 541.815, 542.2)",15.351701,1,Längsschnitt
1,5216742060270992761,Rohr R-K4163S R-K4173S,4816511167455310973,4779752876656844188,5204829332281547346,12.508950,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5048873293262650113,OSM: Knoten 299394923 -> Knoten 4105649557; Lä...,39785520,0,2,0.005,0,350,345.6,Rücklauf,,5216742060270992761,5216742060270992761,False,713616.648712,5.578233e+06,False,5216742060270992761,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713616.649 5578233.372, 713616.465...",4851678476955084637,5469753985021987570,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,0.0,0.004021,27.238,0.340719,0.0,0.0,0.154884,-79.74764,-79.74764,"(-79.74764, -79.74764, -79.74764)",0.197695,0.001937,3.885054,3.953540,3.885055,"(3.95354, 3.9193, 3.885055)",3.953540,3.885055,"(3.95354, 3.9193, 3.885055)",3.953540,3.885055,"(3.95354, 3.9193, 3.885055)",-287.0915,-287.0915,-287.0915,983.7839,983.7833,983.7839,983.7833,"(983.7839, 983.7836, 983.7833)",0.0,12.508950,"(0.0, 6.254475, 12.50895)",59.83229,59.8333,0.311801,0.307780,"(0.311801, 0.3097906, 0.30778)",59.83231,59.83331,"(59.83231, 59.83279, 59.83331)",-0.864134,-0.864133,-0.864134,0.0,0.0,542.20,542.93,"(542.2, 542.565, 542.93)",27.860648,1,Längsschnitt
2,4919359344392474301,Rohr R-K4643S R-K4173S,4816511167455310973,4779752876656844188,5204829332281547346,6.265505,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5715081934973525403,OSM: Knoten 299394922 -> Knoten 4105649557; Lä...,595926249,0,2,0.005,0,350,345.6,Rücklauf,,4919359344392474301,4919359344392474301,Fa

In [50]:
dfs[1][0].head() # Lfdnr 2 VL

,tk,Name,FkdtroRowd,Fkltgr,Fkstrasse,L,Lzu,Rau,Jlambs,Lambda0,Zein,Zaus,Zuml,Asoll,Indschall,Baujahr,Hal,Fkcont,Fk2lrohr,Beschreibung,Idreferenz,Iplanung,Kvr,LineWidthMM,DottedLine,DN,Di,KvrKlartext,HasClosedNSCHs,Tk,Pk,InVariant,Xkor,Ykor,GeometriesDiffer,bz.Fk,bz.Qsvb,bz.Irtrenn,bz.Leckstatus,bz.Leckstart,bz.Leckend,bz.Leckort,bz.Leckmenge,bz.Imptnz,bz.Zvlimptnz,bz.Kantenzv,bz.ITrennWithNSCH,geometry,fkKI,fkKK,PipeTable: Table Name,PipeTable: Name,PipeTable: Fk,PipeTable: Dn,PipeTable: Di,PipeTable: Da,PipeTable: S,PipeTable: Wsteig,PipeTable: Wtiefe,PipeTable: Kt,PipeTable: Pn,PipeTable: Ausfallzeit,PipeTable: Reparatur,PipeTable: Rehabilitation,PipeTable: Tk,PipeTable: Pk,PipeTable: InVariant,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_start,MVEC_end,MVEC_sequence,PDAMPF,PHR,PMIN,PVEC_start,PVEC_end,PVEC_sequence,PVECMAX_INST_start,PVECMAX_INST_end,PVECMAX_INST_sequence,PVECMIN_INST_start,PVECMIN_INST_end,PVECMIN_INST_sequence,QMAV,QMI,QMK,RHOI,RHOK,RHOVEC_start,RHOVEC_end,RHOVEC_sequence,SVEC_start,SVEC_end,SVEC_sequence,TI,TK,TTRVEC_start,TTRVEC_end,TTRVEC_sequence,TVEC_start,TVEC_end,TVEC_sequence,VAV,VI,VK,VOLDA,WVL,ZVEC_start,ZVEC_end,ZVEC_sequence,l_sum,AGSN_Lfdnr,AGSN_Name
0,5691533564979419761,Rohr V-E0 V-K1683S,4816511167455310973,4779752876656844188,5204829332281547346,15.351700,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5025945677694931826,OSM: Knoten 476971238 -> Knoten 299394923; Län...,39785520,0,1,0.005,0,350,345.6,Vorlauf,,5691533564979419761,5691533564979419761,False,713619.921383,5.578219e+06,False,5691533564979419761,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713619.921 5578218.954, 713614.649...",5398100694284104779,4825391580467484032,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,0.0,0.004844,43.12018,0.661968,0.0,0.0,0.152727,79.74764,79.74764,"(79.74764, 79.74764, 79.74764)",0.701074,0.002345,5.800529,5.878635,5.800530,"(5.878635, 5.83958, 5.80053)",5.878635,5.800530,"(5.878635, 5.83958, 5.80053)",5.878635,5.800530,"(5.878635, 5.83958, 5.80053)",287.0915,287.0915,287.0915,965.7,965.7012,965.7000,965.7012,"(965.7, 965.7006, 965.7012)",0.0,15.351700,"(0.0, 7.67585, 15.3517)",89.99999,89.99802,0.000000,0.004844,"(0.0, 0.002422066, 0.004844132)",90.00000,89.99802,"(90.0, 89.99902, 89.99802)",0.880315,0.880315,0.880314,0.0,10082.88,541.49,542.29,"(541.49, 541.89, 542.29)",15.351701,2,Längsschnitt in mlc
1,5048873293262650113,Rohr V-K1683S V-K1693S,4816511167455310973,4779752876656844188,5204829332281547346,12.508950,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5216742060270992761,OSM: Knoten 299394923 -> Knoten 4105649557; Lä...,39785520,0,1,0.005,0,350,345.6,Vorlauf,,5048873293262650113,5048873293262650113,False,713614.648712,5.578233e+06,False,5048873293262650113,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713614.649 5578233.372, 713614.465...",4825391580467484032,5180617780362861593,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,0.0,0.003947,43.58081,0.54515,0.0,0.0,0.152727,79.74764,79.74764,"(79.74764, 79.74764, 79.74764)",0.701027,0.00191,5.732326,5.800530,5.732325,"(5.80053, 5.76643, 5.732325)",5.800530,5.732325,"(5.80053, 5.76643, 5.732325)",5.800530,5.732325,"(5.80053, 5.76643, 5.732325)",287.0915,287.0915,287.0915,965.7012,965.7021,965.7012,965.7021,"(965.7012, 965.7017, 965.7021)",0.0,12.508950,"(0.0, 6.254475, 12.50895)",89.99802,89.9964,0.004844,0.008791,"(0.004844132, 0.006817694, 0.008791257)",89.99802,89.99640,"(89.99802, 89.99722, 89.9964)",0.880314,0.880314,0.880313,0.0,10081.81,542.29,542.99,"(542.29, 542.64, 542.99)",27.860648,2,Längsschnitt in mlc
2,5715081934973525403,Rohr V-K2163S V-K1693S,4816511167455310973,4779752876656844188,5204829332281547346,6.265505,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,4919359344392474301,OSM: Knoten 299394922 -> Knoten 4105649557; Lä...,595926249,0,1,0.005,0,350,345.6,Vorlauf,,5715081934973525403,5715081934973525403,

In [51]:
dfs[1][1].head() # Lfdnr 2 RL

,tk,Name,FkdtroRowd,Fkltgr,Fkstrasse,L,Lzu,Rau,Jlambs,Lambda0,Zein,Zaus,Zuml,Asoll,Indschall,Baujahr,Hal,Fkcont,Fk2lrohr,Beschreibung,Idreferenz,Iplanung,Kvr,LineWidthMM,DottedLine,DN,Di,KvrKlartext,HasClosedNSCHs,Tk,Pk,InVariant,Xkor,Ykor,GeometriesDiffer,bz.Fk,bz.Qsvb,bz.Irtrenn,bz.Leckstatus,bz.Leckstart,bz.Leckend,bz.Leckort,bz.Leckmenge,bz.Imptnz,bz.Zvlimptnz,bz.Kantenzv,bz.ITrennWithNSCH,geometry,fkKI,fkKK,PipeTable: Table Name,PipeTable: Name,PipeTable: Fk,PipeTable: Dn,PipeTable: Di,PipeTable: Da,PipeTable: S,PipeTable: Wsteig,PipeTable: Wtiefe,PipeTable: Kt,PipeTable: Pn,PipeTable: Ausfallzeit,PipeTable: Reparatur,PipeTable: Rehabilitation,PipeTable: Tk,PipeTable: Pk,PipeTable: InVariant,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_start,MVEC_end,MVEC_sequence,PDAMPF,PHR,PMIN,PVEC_start,PVEC_end,PVEC_sequence,PVECMAX_INST_start,PVECMAX_INST_end,PVECMAX_INST_sequence,PVECMIN_INST_start,PVECMIN_INST_end,PVECMIN_INST_sequence,QMAV,QMI,QMK,RHOI,RHOK,RHOVEC_start,RHOVEC_end,RHOVEC_sequence,SVEC_start,SVEC_end,SVEC_sequence,TI,TK,TTRVEC_start,TTRVEC_end,TTRVEC_sequence,TVEC_start,TVEC_end,TVEC_sequence,VAV,VI,VK,VOLDA,WVL,ZVEC_start,ZVEC_end,ZVEC_sequence,l_sum,AGSN_Lfdnr,AGSN_Name
0,5025945677694931826,Rohr R-E0 R-K4163S,4816511167455310973,4779752876656844188,5204829332281547346,15.351700,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5691533564979419761,OSM: Knoten 476971238 -> Knoten 299394923; Län...,39785520,0,2,0.005,0,350,345.6,Rücklauf,,5025945677694931826,5025945677694931826,False,713621.921383,5.578219e+06,False,5025945677694931826,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713621.921 5578218.954, 713616.649...",5160850648779674898,4851678476955084637,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,0.0,0.004935,26.63305,0.408863,0.0,0.0,0.154884,-79.74764,-79.74764,"(-79.74764, -79.74764, -79.74764)",0.197685,0.002378,3.953543,4.025455,3.953540,"(4.025455, 3.9895, 3.95354)",4.025455,3.953540,"(4.025455, 3.9895, 3.95354)",4.025455,3.953540,"(4.025455, 3.9895, 3.95354)",-287.0915,-287.0915,-287.0915,983.7845,983.7839,983.7845,983.7839,"(983.7845, 983.7842, 983.7839)",0.0,15.351700,"(0.0, 7.67585, 15.3517)",59.83105,59.83229,0.316736,0.311801,"(0.3167359, 0.3142685, 0.311801)",59.83105,59.83231,"(59.83105, 59.83167, 59.83231)",-0.864133,-0.864133,-0.864133,0.0,0.0,541.43,542.20,"(541.43, 541.815, 542.2)",15.351701,2,Längsschnitt in mlc
1,5216742060270992761,Rohr R-K4163S R-K4173S,4816511167455310973,4779752876656844188,5204829332281547346,12.508950,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5048873293262650113,OSM: Knoten 299394923 -> Knoten 4105649557; Lä...,39785520,0,2,0.005,0,350,345.6,Rücklauf,,5216742060270992761,5216742060270992761,False,713616.648712,5.578233e+06,False,5216742060270992761,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713616.649 5578233.372, 713616.465...",4851678476955084637,5469753985021987570,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,0.0,0.004021,27.238,0.340719,0.0,0.0,0.154884,-79.74764,-79.74764,"(-79.74764, -79.74764, -79.74764)",0.197695,0.001937,3.885054,3.953540,3.885055,"(3.95354, 3.9193, 3.885055)",3.953540,3.885055,"(3.95354, 3.9193, 3.885055)",3.953540,3.885055,"(3.95354, 3.9193, 3.885055)",-287.0915,-287.0915,-287.0915,983.7839,983.7833,983.7839,983.7833,"(983.7839, 983.7836, 983.7833)",0.0,12.508950,"(0.0, 6.254475, 12.50895)",59.83229,59.8333,0.311801,0.307780,"(0.311801, 0.3097906, 0.30778)",59.83231,59.83331,"(59.83231, 59.83279, 59.83331)",-0.864134,-0.864133,-0.864134,0.0,0.0,542.20,542.93,"(542.2, 542.565, 542.93)",27.860648,2,Längsschnitt in mlc
2,4919359344392474301,Rohr R-K4643S R-K4173S,4816511167455310973,4779752876656844188,5204829332281547346,6.265505,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5715081934973525403,OSM: Knoten 299394922 -> Knoten 4105649557; Lä...,595926249,0,2,0.005,0,350,345.6,Rücklauf,,4919359344392474301,49193593